# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


### Ce qui a ete change

Consigne : changer l'objectif du planner, et adapter prompts, outils et agents en consequence.

**Nouvel objectif** — une veille technologique que je pourrais reellement avoir a produire :
mapper l'etat de l'art de la **detection automatique de crises d'epilepsie par IA** en unite de
monitoring video-EEG, et en tirer les questions a poser avant un pilote.

| Element | Avant | Apres |
|---|---|---|
| `OBJECTIVE` | "Write a weather report for SF today" | briefing structure sur la detection de crises par IA (acteurs, statut reglementaire, questions pour un pilote) |
| Outils | `Search`, `TODO` | `Search`, `TODO`, **`Glossary`** (lexique deterministe du domaine), **`SaveNote`** (ecrit les constats dans un fichier) |
| Prompt du `TODO` | generique | plan structure selon 4 axes : clinique, technique, reglementaire, operationnel |
| Prompt de l'agent (`prefix`) | "You are an AI who performs one task" | role d'analyste clinique + regles explicites : citer les URL, distinguer une affirmation commerciale d'une donnee publiee, dire "unknown" plutot qu'inventer |
| `max_iterations` | 3 | 5 |

**Pourquoi ces deux outils en plus** — c'est le point interessant du lab. `Search` renvoie du texte
non verifie, et BabyAGI n'a par defaut aucune memoire exploitable en dehors du vectorstore.
`Glossary` est **deterministe** : sur des sigles metier (SEEG, EMU, MDR, 510(k)), une reponse fiable
vaut mieux qu'une recherche web approximative. `SaveNote` donne a l'agent un **effet de bord utile** :
a la fin du run, il reste un fichier `briefing_notes.md` exploitable, au lieu d'un historique de
console.

> **Prerequis d'execution** : un fichier `.env` avec `OPENAI_API_KEY` et `SERPAPI_API_KEY`.
> Attention au cout : chaque iteration declenche plusieurs appels OpenAI + SerpAPI.

## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [ ]:
!pip install langchain_community langchain_experimental langchain_openai

In [5]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [7]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [8]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [9]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [ ]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

# ---------------------------------------------------------------- outil 1 : TODO
# Prompt du planner adapte au domaine : on impose les axes d'analyse, sinon le
# modele produit une todo list generique ("research the topic, write the report").
todo_prompt = PromptTemplate.from_template(
    """You are a planner who is an expert at coming up with a todo list for a given objective.

The objective concerns the clinical evaluation of an AI medical technology.
Structure the todo list along these four axes, and keep each item concrete and verifiable:
1. Clinical: what the device claims to detect, on which population, with which reported
   sensitivity / false-alarm rate, and in which published studies.
2. Technical: signal modality (scalp EEG, SEEG, ECG, accelerometry), latency, integration
   with existing monitoring systems.
3. Regulatory: CE marking under MDR, FDA clearance (510(k) or De Novo), intended use as stated
   in the official documentation.
4. Operational: impact on nursing workflow, alarm handling, training, and what should be
   measured during a pilot.

Come up with a todo list for this objective: {objective}"""
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)

# -------------------------------------------------------------- outil 2 : Search
search = SerpAPIWrapper()

# ------------------------------------------------------------ outil 3 : Glossary
# Outil deterministe : sur des sigles metier, une definition fiable vaut mieux
# qu'une recherche web interpretee par le modele.
GLOSSARY = {
    "emu": "Epilepsy Monitoring Unit: hospital unit where patients are recorded on continuous video-EEG for several days to characterise their seizures.",
    "veeg": "Video-EEG: simultaneous recording of scalp EEG and video, the reference method to classify seizures.",
    "seeg": "Stereo-electroencephalography: intracranial recording with depth electrodes, used in presurgical evaluation of drug-resistant focal epilepsy.",
    "ictal": "Relating to the seizure itself, as opposed to interictal (between seizures).",
    "interictal": "Between seizures. Interictal epileptiform discharges are markers of epilepsy but are not seizures.",
    "fbtcs": "Focal to bilateral tonic-clonic seizure: the seizure type most consistently detected by automated systems.",
    "sudep": "Sudden Unexpected Death in Epilepsy: the main safety rationale invoked for nocturnal seizure detection devices.",
    "ilae": "International League Against Epilepsy: publishes the reference classification of seizures and epilepsies.",
    "mdr": "EU Medical Device Regulation 2017/745: the regulation under which a device must be CE marked to be placed on the EU market.",
    "510k": "US FDA premarket notification: clearance route based on substantial equivalence to an already marketed device. Clearance is not the same as approval.",
    "de novo": "US FDA route for a novel device with no predicate, creating a new classification.",
    "sensitivity": "Proportion of real events correctly detected. In seizure detection it is always reported together with the false alarm rate per 24 hours: one number without the other is meaningless.",
    "far": "False Alarm Rate, usually expressed per 24 hours. The metric that decides whether a detection system is usable at the bedside.",
}


def glossary_lookup(term: str) -> str:
    """Return the definition of a domain acronym, or a list of known terms."""
    cle = term.strip().lower().replace("(", "").replace(")", "").replace("-", "").replace(".", "")
    for k, v in GLOSSARY.items():
        if k in cle:
            return f"{k.upper()}: {v}"
    return ("Unknown term. Known terms: " + ", ".join(sorted(GLOSSARY)))


# ------------------------------------------------------------ outil 4 : SaveNote
# Effet de bord utile : a la fin du run il reste un fichier exploitable,
# et non seulement un historique de console.
NOTES_FILE = "briefing_notes.md"


def save_note(note: str) -> str:
    """Append a finding to the briefing notes file."""
    note = note.strip()
    if not note:
        return "Nothing to save: empty note."
    with open(NOTES_FILE, "a", encoding="utf-8") as f:
        f.write(f"- {note}\n")
    return f"Saved to {NOTES_FILE} ({len(note)} characters)."


tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events, companies, products or published studies. Input: a search query.",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
    Tool(
        name="Glossary",
        func=glossary_lookup,
        description="useful when you encounter a clinical or regulatory acronym from epilepsy monitoring (EMU, vEEG, SEEG, ictal, SUDEP, ILAE, MDR, 510k, sensitivity, FAR). Input: the single term to define. Prefer this over Search for definitions.",
    ),
    Tool(
        name="SaveNote",
        func=save_note,
        description="useful for recording a finding that must appear in the final briefing. Input: one factual sentence, with its source URL when it comes from Search.",
    ),
]

# Prompt de l'agent : role explicite + regles de fiabilite. Sans ces regles,
# l'agent recopie les arguments commerciaux des sites des fabricants.
prefix = """You are a clinical AI analyst working in an epilepsy monitoring unit.
You perform one task based on the following objective: {objective}.
Take into account these previously completed tasks: {context}.

Rules you must follow:
- Always give the source URL for any factual claim obtained through Search.
- Clearly separate what a vendor claims from what has been published in a peer-reviewed study.
- Report detection performance only as a pair: sensitivity AND false alarm rate per 24h.
- If you cannot verify something, write "unknown" instead of guessing.
- Record every important finding with the SaveNote tool."""

suffix = """Question: {task}
{agent_scratchpad}"""

prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

print(prompt.template[:1200], "...")

In [12]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ZeroShotAgent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use create_react_agent instead.
  warn_deprecated(


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [ ]:
OBJECTIVE = (
    "Produce a short briefing on the current landscape of AI-based automatic seizure detection "
    "for epilepsy monitoring units: which systems exist, what detection performance is actually "
    "published for them, what their regulatory status is in the EU and the US, and what a nursing "
    "team should verify before running a pilot."
)
print(OBJECTIVE)

In [ ]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
# 5 iterations : assez pour voir la liste de taches se reorganiser, sans faire exploser
# le cout (chaque iteration = plusieurs appels OpenAI + SerpAPI).
max_iterations: Optional[int] = 5
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [16]:
baby_agi({"objective": OBJECTIVE})

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should create a todo list based on the given objective
Action: TODO
Action Input: Create a todo list for SF weather report
Observation: 

1. Research the current weather patterns in San Francisco.
2. Determine the sources for obtaining accurate weather information for the city.
3. Create a list of key weather elements to include in the report, such as temperature, precipitation, wind speed, and humidity.
4. Decide on the format of the report, whether it will be a written document, a presentation, or a combination of both.
5. Set a timeline for when the report needs to be completed and shared.
6. Gather data from reliable sources, such as the National Weather Service or local meteorologists.
7. Organize the data into a clear and easy-to-understand format.
8. Include any relevant graphics or visuals to enhance the report.
9. Proofread and edit the report 

{'objective': 'Write a weather report for SF today'}

In [ ]:
# Ce que l'agent a effectivement retenu via l'outil SaveNote
import os

if os.path.exists(NOTES_FILE):
    print(open(NOTES_FILE, encoding="utf-8").read())
else:
    print("Aucune note : l'agent n'a pas utilise l'outil SaveNote pendant ce run.")

---

## Reflexion

**Ce que le changement d'objectif rend visible.** L'objectif d'origine ("weather report for SF")
se resout en un appel de recherche : la boucle BabyAGI n'y sert pratiquement a rien. Un objectif de
veille a plusieurs axes (clinique / technique / reglementaire / operationnel) est le cas ou la
decomposition en taches a un sens — et ou ses faiblesses deviennent visibles aussi.

**Les limites de BabyAGI observees sur ce type d'objectif :**

- **Pas de critere d'arret.** BabyAGI s'arrete sur `max_iterations`, pas parce que l'objectif est
  atteint. Rien dans l'architecture ne compare l'etat courant a l'objectif : la boucle genere de
  nouvelles taches indefiniment.
- **Derive de la liste de taches.** La chaine de creation de taches part du resultat de la derniere
  tache. Sur un sujet large, elle produit des taches de plus en plus specifiques et de plus en plus
  eloignees de l'objectif initial, et souvent des quasi-doublons.
- **Le contexte passe est maigre.** Seul un resume des taches precedentes est reinjecte via
  `{context}` : l'agent refait des recherches deja faites. C'est precisement le trou que l'outil
  `SaveNote` comble ici, en materialisant un etat persistant hors du vectorstore.
- **Aucune verification de ce que renvoie `Search`.** SerpAPI rend le contenu des pages, y compris
  les pages commerciales des fabricants. Les regles ajoutees dans le `prefix` (citer l'URL,
  distinguer claim et etude publiee, exiger sensibilite *et* taux de fausses alarmes) reduisent le
  probleme mais ne le suppriment pas : c'est une consigne, pas une contrainte.

**Ce que je retiens en tant qu'AI engineering.** Sur un sujet a enjeu clinique, un agent autonome de
ce type est utile pour **degrossir** — trouver les acteurs, rassembler des liens, structurer un plan.
Il n'est pas fiable pour conclure : les chiffres de performance et les statuts reglementaires
doivent etre reverifies a la source. La valeur du systeme vient moins de l'autonomie de la boucle que
de la qualite des outils qu'on lui donne — et le seul outil vraiment fiable de ce notebook est celui
qui ne fait appel a aucun LLM.